In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np

# --- 1. CONFIGURATION & DEVICE SETUP ---
DATA_DIR = os.getcwd()  # Or specify the path to your folder containing images and CSV
CSV_PATH = os.path.join(DATA_DIR, 'GroundTruth.csv')

# Ensure Mac multiprocessing safety by keeping num_workers=0 inside DataLoader
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.0001
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"🚀 Initializing Training Pipeline on device: {DEVICE}")

# --- 2. CUSTOM DATASET CLASS ---
class HAM10000Dataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform
        
        # Define the 7 categorical class columns from GroundTruth.csv
        self.class_columns = ['AKIEC', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'VASC']
        
        # Convert one-hot encoded columns back to a single categorical index (0 to 6) using np.argmax
        self.labels = np.argmax(self.df[self.class_columns].values, axis=1)
        
        # Locate image paths dynamically across common subfolders
        self.img_paths = []
        for img_id in self.df['image']:
            found = False
            for subfolder in ['HAM10000_images_part_1', 'HAM10000_images_part_2', '.']:
                full_path = os.path.join(self.img_dir, subfolder, f"{img_id}.jpg")
                if os.path.exists(full_path):
                    self.img_paths.append(full_path)
                    found = True
                    break
            if not found:
                # Fallback path structure
                self.img_paths.append(os.path.join(self.img_dir, f"{img_id}.jpg"))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

# --- 3. TRANSFORMS ---
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 4. DATA LOADER & CLASS WEIGHTING ---
dataset = HAM10000Dataset(csv_file=CSV_PATH, img_dir=DATA_DIR, transform=train_transforms)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# Compute Class Counts & Inverse Frequency Weights to fix dataset skew
class_counts = np.bincount(dataset.labels)
total_samples = len(dataset)
num_classes = len(class_counts)
class_weights = total_samples / (num_classes * class_counts.astype(float))
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

print(f"📊 Dataset Class Distribution: {class_counts}")
print(f"⚖️ Computed Class Weights: {class_weights_tensor}")

# --- 5. MODEL ARCHITECTURE SETUP ---
# Using MobileNetV3-Small with updated weights parameter to avoid deprecation warnings
weights = models.MobileNet_V3_Small_Weights.DEFAULT
model = models.mobilenet_v3_small(weights=weights)

# Modify final classifier layer for 7 output classes
model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
model = model.to(DEVICE)

# Loss function with Inverse Frequency Class Weights applied
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# --- 6. TRAINING LOOP ---
if __name__ == '__main__':
    print("🎯 Starting Training Loop across Epochs...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct_preds = 0
        total_preds = 0

        for images, labels in dataloader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct_preds += (predicted == labels).sum().item()
            total_preds += labels.size(0)

        epoch_loss = running_loss / total_preds
        epoch_acc = (correct_preds / total_preds) * 100
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

    # Save trained model weights
    torch.save(model.state_dict(), 'ham10000_skin_model.pth')
    print("🎉 Training finished! Model successfully saved as 'ham10000_skin_model.pth'")